# Twinkle 推理示例（Inference with LoRA）

> 🎯 **本 Notebook 目标**：对训练好的 LoRA 模型进行 **在线推理**，验证训练效果。

本 Notebook 演示如何使用 Twinkle **线上服务** 对训练好的 LoRA 模型进行推理。无需本地 GPU。

### 整体流程

```
初始化客户端 → 加载 LoRA 检查点 → 构造 Prompt → 在线采样 → 输出结果
```

### 前置条件

| 条件 | 说明 |
|------|------|
| 环境变量 | 设置 `MODELSCOPE_TOKEN` |
| LoRA 检查点 | 训练产出的 `twinkle://` 路径 |
| 依赖安装 | `pip install twinkle-kit[tinker]` |
> 💡 **获取 Token**：访问 [ModelScope Token 页面](https://www.modelscope.cn/my/access/token) 获取你的 `MODELSCOPE_TOKEN`，并设置为环境变量：`export MODELSCOPE_TOKEN=<你的Token>`


> ⚠️ **MoE 模型在线推理限制**：由于 vLLM 目前不支持 Expert LoRA，
> 在线推理无法直接加载 MoE 模型的 LoRA 权重。
> 请先将训练好的检查点上传到 ModelScope Hub（参见训练 Notebook 中的上传步骤），
> 再通过「合并权重并导出」得到完整模型后部署使用。
> 
> 本 Notebook 中的在线推理示例适用于非 MoE 模型，或已合并权重的模型。

## 🚀 线上推理服务

本 Notebook 通过 **ModelScope 线上服务** 进行推理，你的 Notebook 环境不需要 GPU。

### 架构示意图

```
┌───────────────────────────────────────────────────────┐
│              你的 Notebook（CPU 环境）                    │
│                                                       │
│  ┌──────────────┐    HTTP     ┌────────────────────┐  │
│  │  Tinker       │ ─────────► │  ModelScope 云端    │  │
│  │  ServiceClient│ ◄───────── │  推理集群           │  │
│  └──────────────┘   生成结果   │                    │  │
│       │                       │  ┌────┐ ┌────┐     │  │
│       │  发送 Prompt           │  │GPU0│ │GPU1│ ... │  │
│       │  接收生成文本          │  └────┘ └────┘     │  │
│       ▼                       │  基座模型 + LoRA    │  │
│  ┌──────────────┐             └────────────────────┘  │
│  │  Template     │  本地编码 Prompt / 解码结果         │
│  └──────────────┘                                     │
└───────────────────────────────────────────────────────┘
```

> 🔗 本项目由 [Twinkle](https://github.com/modelscope/twinkle) 框架提供支持 | [GitHub](https://github.com/modelscope/twinkle)

## 第一步：导入依赖与配置

| 配置项 | 说明 |
|--------|------|
| `BASE_MODEL` | 基座模型 ID |
| `weight_path` | 训练产出的 LoRA 检查点路径（`twinkle://...` 格式） |
| `MODELSCOPE_TOKEN` | ModelScope API Token（环境变量） |

## 环境安装

首次运行前，请先执行以下安装命令。如已安装可跳过此步。

In [ ]:
!pip install twinkle-kit[tinker] -q

In [ ]:
import os
from tinker import types
from twinkle import init_tinker_client, get_logger
from twinkle.data_format import Message, Trajectory
from twinkle.template import Template

logger = get_logger()

BASE_MODEL = 'Qwen/Qwen3.6-35B-A3B'

# TODO: 替换为你的训练检查点路径
weight_path = '<替换为你的 twinkle:// 检查点路径>'  # 例如: 'twinkle://xxx/weights/twinkle-lora-2'

## 第二步：初始化客户端

连接 ModelScope 线上推理服务，并加载训练好的 LoRA 检查点。

> **重要**：必须先调用 `init_tinker_client()` 完成运行时初始化，再 import `ServiceClient`。

In [ ]:
init_tinker_client()
from tinker import ServiceClient

service_client = ServiceClient(
    base_url='http://www.modelscope.cn/twinkle',
    api_key=os.environ.get('MODELSCOPE_TOKEN'),
)

# 加载 LoRA 检查点并创建采样客户端
sampling_client = service_client.create_sampling_client(
    model_path=weight_path,
    base_model=BASE_MODEL,
)
print('采样客户端创建成功')

## 第三步：构造 Prompt

使用 Template 将对话格式的 Trajectory 编码为 token 序列。

> Template 需要在本地加载 tokenizer（会自动从 ModelScope 下载），但 **不需要 GPU**。

In [ ]:
template = Template(model_id=f'ms://{BASE_MODEL}')

# 构造多条 Prompt
prompts = [
    Trajectory(
        messages=[
            Message(role='system', content='You are a helpful assistant.'),
            Message(role='user', content='什么是强化学习？请简单解释。'),
        ]
    ),
    Trajectory(
        messages=[
            Message(role='user', content='Write a short poem about the moon.'),
        ]
    ),
    Trajectory(
        messages=[
            Message(role='user', content='求解方程 2x + 3 = 11，x 等于多少？'),
        ]
    ),
]

print(f'共 {len(prompts)} 条 Prompt')

## 第四步：采样推理

对每条 Prompt 编码后发送到线上服务进行采样。

| 参数 | 说明 |
|------|------|
| `max_tokens` | 最大生成 token 数 |
| `temperature` | 采样温度，越高越多样 |
| `num_samples` | 每条 Prompt 生成几条回答 |

In [ ]:
params = types.SamplingParams(
    max_tokens=256,
    temperature=0.7,
)

for i, trajectory in enumerate(prompts):
    # 编码 Prompt
    input_feature = template.encode(trajectory, add_generation_prompt=True)
    input_ids = input_feature['input_ids'].tolist()
    prompt = types.ModelInput.from_ints(input_ids)

    # 发送采样请求
    future = sampling_client.sample(prompt=prompt, sampling_params=params, num_samples=1)
    result = future.result()

    # 解码并输出
    user_msg = trajectory['messages'][-1]['content']
    print(f'\n{"="*60}')
    print(f'Prompt {i}: {user_msg}')
    print(f'{"─"*60}')
    for j, seq in enumerate(result.sequences):
        print(f'{template.decode(seq.tokens)}')

## 合并权重并导出

训练得到的 LoRA 权重可以与原始模型合并，导出为完整的 HuggingFace 模型，方便后续部署和推理。

> **注意**：合并操作需要 GPU 资源（需要加载完整模型），请在有足够显存的环境下执行。

```bash
CUDA_VISIBLE_DEVICES=0,1,2,3 \
NPROC_PER_NODE=4 \
/opt/conda/envs/twinkle/bin/megatron export \
    --model Qwen/Qwen3.6-35B-A3B \
    --adapters <替换为你的 LoRA 检查点路径> \
    --output_dir <替换为输出目录> \
    --merge_lora true \
    --to_hf true \
    --tensor_model_parallel_size 2 \
    --expert_model_parallel_size 2 \
    --pipeline_model_parallel_size 2
```

合并完成后，输出目录中即为完整的 HuggingFace 模型，可直接用于推理或部署。

## 常见问题

| 问题 | 可能原因 | 解决方法 |
|------|----------|----------|
| 连接超时 | 网络问题或服务端繁忙 | 检查网络并重试 |
| 检查点不存在 | weight_path 路径错误 | 确认训练完成并检查 save_result.path |
| 输出质量差 | LoRA 训练不充分 | 增加训练步数或调整学习率 |
| Token 认证失败 | MODELSCOPE_TOKEN 未设置 | 检查环境变量 |